# Stage 10 — Accuracy vs Ground Truth

First stage that reads **current-stay** `ground_truth.json`.

For each admission, compare:

1. **Top DiffDx** (Stage 8 / Stage 9) vs GT primary title (exact + MiniLM ≥ 0.70)
2. **Primary ICD** — exact, 3-character family, and MiniLM title match ≥ 0.70
3. **Code set** — exact P/R/F1 **and** semantic P/R/F1 (same meaning, different code)
4. Related near-misses (sim 0.50–0.70) shown but not counted

Stage 8 (map-only) and Stage 9 (LLM-confirmed) are scored side by side.

**Output:**
- per admission: `accuracy/comparison.txt` + `accuracy/accuracy.json`
- cohort: `data/stage_10_evaluation/accuracy_summary.json` + `cohort_metrics.txt`
- full-text tables: `data/stage_10_evaluation/primary_match_table.csv` (DiffDx rank + GT seq)

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

from pipeline import (
    EVAL_COHORT_TXT,
    EVAL_SUMMARY_JSON,
    EXPORT_DIR,
    STAGE_10_DIR,
    print_pipeline_banner,
    run_stage10_evaluation,
)

print_pipeline_banner()
STAGE_10_DIR.mkdir(parents=True, exist_ok=True)
print(f"Export dir : {EXPORT_DIR}")
print(f"Stage 10 out: {STAGE_10_DIR}")
print("Requires Stage 8 icd_coding.json (+ Stage 9 icd_coding_confirmed.json if run).")

In [ ]:
payload = run_stage10_evaluation(export_dir=EXPORT_DIR, use_embeddings=True)
summary = payload.get("summary") or {}
s8 = summary.get("stage8") or {}
s9 = summary.get("stage9") or {}
print(f"\nAdmissions: {summary.get('n_admissions')} | with Stage 9: {summary.get('n_with_stage9')}")
print(f"{'metric':<32} {'Stage 8':>10} {'Stage 9':>10}")
print("-" * 54)
for label, key in [
    ("Top dx semantic match", "dx_semantic_match_rate"),
    ("Primary ICD exact", "primary_icd_exact_rate"),
    ("Primary ICD family", "primary_icd_family_rate"),
    ("Primary ICD semantic", "primary_icd_semantic_rate"),
    ("Mean ICD F1 (exact)", "mean_f1"),
    ("Mean ICD F1 (semantic)", "mean_semantic_f1"),
]:
    print(f"{label:<32} {s8.get(key, 0):>10.3f} {s9.get(key, 0):>10.3f}")
print(f"\nSaved → {EVAL_SUMMARY_JSON}")
print(f"Cohort → {EVAL_COHORT_TXT}")
print(f"Per patient → {EXPORT_DIR}/patient_*/admissions/hadm_*/accuracy/")

In [ ]:
from pathlib import Path

sample = next(Path(EXPORT_DIR).glob("patient_*/admissions/hadm_*/accuracy/comparison.txt"), None)
if sample:
    print(sample)
    print(sample.read_text(encoding="utf-8")[:2500])
else:
    print("No comparison.txt yet — run the cell above.")